# Lab 1. Analysis of Cereal Yield and Climate Data


###1. Git Repository Setup

Connected the Databricks workspace to a Git repository using a Git folder. The notebook was created and maintained in the repository to keep the work version-controlled and allow changes to be tracked.



### 2. Shared Cluster Setup

Attached the Databricks notebook to the provided shared cluster and executed the notebook using the available compute resource.

### 3. Git Commit

Committed the completed notebook changes to the connected Git repository from the beginning of the project to keep the implementation version-controlled.

### 4. Data Ingestion


Reading the static historical dataset containing cereal production, yield, and climate indicators for European countries from 1990 to 2022 (sources: FAOSTAT (wheat, barley, and maize) and World Bank Climate Change Knowledge Portal (CCKP ERA5)(climate indicators )). The dataset was loaded into a Spark DataFrame for further processing and analysis.

In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbr_dev_ua5816bd/viktoriia_kalenichenko/raw_data/europe_cereal_yield_climate_1990_2022.csv")

df = df.filter(df["AREA"] != "Russian Federation")

In [0]:
display(df)

In [0]:
df.printSchema()

### 5. Delta Tables Creation


After cleaning and preparing the data, I saved the processed dataset as Delta tables in Unity Catalog for persistent storage and further SQL analysis. I then divided the data into two smaller tables: "cereals" containing agricultural data and "climate№ containing climate-related indicators. Additionally, I added two external datasets: "climate_factors_df", which serves as a dictionary for interpreting the climate indicator abbreviations, and "environment", which provides additional information about environmental factors that may affect cereal yield. Then all of these dataframes were updated to delta tables.

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate")

In [0]:
#select operation + craetion tables
cereals_df = df.select(
    "AREA",
    "ITEM",
    "YEAR",
    "AREA_HARVESTED",
    "PRODUCTION_QUANTITY",
    "YIELD"
)

cereals_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.cereals")

In [0]:
climate_df = df.select(
    "AREA",
    "YEAR",
    "WB_CCKP_CDD",
    "WB_CCKP_CDD65",
    "WB_CCKP_CSDI",
    "WB_CCKP_CWD",
    "WB_CCKP_FD",
    "WB_CCKP_HD30",
    "WB_CCKP_HD35",
    "WB_CCKP_HD40",
    "WB_CCKP_HD42",
    "WB_CCKP_HD45",
    "WB_CCKP_HD50",
    "WB_CCKP_HDD65",
    "WB_CCKP_HI35",
    "WB_CCKP_HI37",
    "WB_CCKP_HI39",
    "WB_CCKP_HI41",
    "WB_CCKP_HURS",
    "WB_CCKP_ID",
    "WB_CCKP_PR",
    "WB_CCKP_R20MM",
    "WB_CCKP_R50MM",
    "WB_CCKP_R95PTOT",
    "WB_CCKP_RX1DAY",
    "WB_CCKP_RX5DAY",
    "WB_CCKP_SD",
    "WB_CCKP_TAS",
    "WB_CCKP_TASMAX",
    "WB_CCKP_TASMIN",
    "WB_CCKP_TNN",
    "WB_CCKP_TR",
    "WB_CCKP_TR23",
    "WB_CCKP_TR26",
    "WB_CCKP_TR29",
    "WB_CCKP_TR32",
    "WB_CCKP_TX84RR",
    "WB_CCKP_TXX",
    "WB_CCKP_WSDI"
)

climate_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.climate")

In [0]:
climate_factors_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbr_dev_ua5816bd/viktoriia_kalenichenko/raw_data/climatic_factors.csv")

climate_factors_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.climatic_factors")

display(climate_factors_df)

In [0]:
environment = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbr_dev_ua5816bd/viktoriia_kalenichenko/raw_data/environment_clean.csv")

environment.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbr_dev_ua5816bd.viktoriia_kalenichenko.environment")

display(environment)

### 6. Spark DataFrame Operations

In [0]:
#filter operation
display(df.filter(df["YEAR"] >= 2020))

In [0]:
#groupBy operation
display(df.groupBy("AREA").avg("YIELD"))

In [0]:
joined_df = cereals_df \
    .join(
        climate_df,
        on=["AREA", "YEAR"],
        how="inner"
    ) \
    .join(
        environment,
        on=["AREA", "YEAR"],
        how="left"
    )

display(joined_df)

In [0]:
#other operation + showing which countries we analyze
display(df.select("AREA").distinct().orderBy("AREA"))

In [0]:
#checking for dublicates 

joined_df.groupBy("AREA", "YEAR") \
    .count() \
    .filter("count > 1") \
    .display()

After performing the ".join" operation, a Delta table named "joined_data" was created to enable further work using SQL queries.

In [0]:
joined_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbr_dev_ua5816bd.viktoriia_kalenichenko.joined_data"
    )

### 7. Data Analysis & Dashboard

Using SQL and Spark operations to analyse cereal yield and its relationship with climate and environmental indicators such as temperature, precipitation, and water resources.
Creating a basic Databricks dashboard with visualizations based on important information from data analysis.

_This query provides a quick overview of the main data used for further analysis. To avoid processing outdated and unnecessary information, the dataset was limited to the most recent available years, 2020–2022. The query calculates the average annual yield for each of the three cereal crops under study (barley, wheat, and maize) — across the selected European countries. This allows us to compare cereal productivity between countries and identify differences in average yield.

In [0]:
%sql
-- queries for the dashboard
SELECT
    AREA,
    ITEM,
    YEAR,
    YIELD
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.cereals
WHERE YEAR BETWEEN 2020 AND 2022
ORDER BY AREA, ITEM, YEAR;

Databricks visualization. Run in Databricks to view.


This query calculates the average cereal yield for each selected European country. The value is calculated as the average yield for the most recent available years, 2020–2022, and includes all three cereal crops under study: barley, wheat, and maize. Averaging data across several years and different crops may reduce the precision of the comparison for a specific crop or year. However, this approach is sufficient for providing a general overview of cereal productivity across the selected countries.


In [0]:
%sql

SELECT
    AREA,
    country_code,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.joined_data
WHERE YEAR BETWEEN 2020 AND 2022
GROUP BY AREA, country_code
ORDER BY avg_yield DESC



Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.


This query analyses the relationship between average temperature and cereal yield. Two visualizations were created to examine this relationship from different perspectives. The first visualization compares temperature and average yield across the selected European countries, while the second focuses on the three cereal crops separately. This approach helps to observe whether changes in average temperature are associated with differences in cereal yield between countries and crops.

In [0]:
%sql
SELECT
    AREA,
    YEAR,
    AVG(WB_CCKP_TAS) AS avg_temperature,
    AVG(YIELD) AS avg_yield
FROM  dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate
GROUP BY AREA, YEAR
ORDER BY AREA, YEAR;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    AREA,
    YEAR,
    ITEM,
    AVG(WB_CCKP_TAS) AS avg_temperature,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate
GROUP BY AREA, YEAR, ITEM
ORDER BY AREA, YEAR, ITEM;

Databricks visualization. Run in Databricks to view.


This query shows the relationship between precipitation and grain crop yields for each crop separately. The visualizations allow us to observe how precipitation levels are associated with the yield of barley, wheat, and maize and to compare the patterns between the three crops.

In [0]:
%sql

SELECT
    AREA,
    YEAR,
    ITEM,
    AVG(WB_CCKP_PR) AS avg_precipitation,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.cereal_climate
GROUP BY AREA, YEAR, ITEM
ORDER BY AREA, YEAR, ITEM;

Databricks visualization. Run in Databricks to view.


This query analyses the relationship between environmental water resources and average cereal yield across the selected European countries. For each country, it calculates the average amount of renewable freshwater resources per capita, the average share of freshwater withdrawals, and the average cereal yield. The results are grouped by country and sorted by average yield in descending order, making it possible to compare countries with different levels of water resources and cereal productivity.


In [0]:
%sql

SELECT
    AREA,
    AVG(freshwater_per_capita) AS avg_freshwater_per_capita,
    AVG(freshwater_withdrawals) AS avg_freshwater_withdrawals,
    AVG(YIELD) AS avg_yield
FROM dbr_dev_ua5816bd.viktoriia_kalenichenko.joined_data
GROUP BY AREA
ORDER BY avg_yield DESC;

### 8. External API Integration

Fetching agricultural economic data from the World Bank External API. The selected indicator shows agriculture, forestry, and fishing value added as a percentage of GDP, which can be used for further analysis. Firstly, I collect and print all the ISO country codes required for the API request and combine them into a single list that will be included in the API URL.

In [0]:
country_codes = [
    row["country_code"]
    for row in environment
        .select("country_code")
        .distinct()
        .collect()
]

country_codes = [code for code in country_codes if code]

print(country_codes)

countries = ";".join(country_codes)

print(countries)

In [0]:
#external API
import requests

url = "https://api.worldbank.org/v2/country/LVA;POL;FRA;ITA;UKR;HRV;GBR;MLT;BLR;SVK;HUN;NOR;FIN;ALB;BIH;NLD;LUX;MNE;AUT;PRT;LTU;ROU;DNK;ESP;EST;IRL;SWE;SVN;GRC;BEL;MKD;DEU;MDA;BGR;SRB;CHE;CZE;ISL/indicator/NV.AGR.TOTL.ZS"

response = requests.get(
    url,
    params={
        "format": "json",
        "date": "1990:2022"
    }
)

data = response.json()

print(data)

Created a dataframe for it.

In [0]:
import pandas as pd

rows = data[1]

api_df = pd.DataFrame([
    {
        "AREA": row["country"]["value"],
        "country_code": row["countryiso3code"],
        "YEAR": int(row["date"]),
        "agriculture_value_added_gdp": row["value"]
    }
    for row in rows
    if row["value"] is not None
])

display(api_df)

In [0]:
agriculture_gdp = spark.createDataFrame(api_df)

display(agriculture_gdp)

agriculture_gdp.printSchema()

Update it to the delta table and add to my path.

In [0]:
agriculture_gdp.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbr_dev_ua5816bd.viktoriia_kalenichenko.agriculture_gdp"
    )

###9. Optional Addition


Delta Lake was used to store the processed datasets as reliable and persistent Delta tables in Unity Catalog. It provides several advantages for data processing and analysis:

ACID Transactions: Ensure that data changes are completed safely and prevent tables from being left in an inconsistent state if a write operation fails.
Time Travel: Keeps previous versions of the table, allowing users to access historical data versions and recover from accidental changes.
Schema Enforcement and Evolution: Helps ensure that incoming data matches the expected table structure while allowing the schema to be safely updated when necessary.
